# 🌸 Laboratório de Aprendizado Federado Canônico (Flower + PyTorch)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JM3L0/flower_puro/blob/main/executar_flower_puro.ipynb)

Este notebook executa o ciclo completo de **Aprendizado Federado Canônico (Flower Puro)** sobre o dataset CIFAR-10 sincronizando diretamente com o repositório oficial: **[JM3L0/flower_puro](https://github.com/JM3L0/flower_puro)**.

---
### ⚡ Configuração Inicial de GPU:
Para acelerar o treinamento na GPU T4 gratuita do Colab, selecione no menu superior:
**Ambiente de execução (Runtime) -> Alterar tipo de ambiente de execução -> Acelerador de hardware: T4 GPU -> Salvar**.

## 1. Verificar Disponibilidade da GPU T4

In [ ]:
!nvidia-smi

## 2. Clonar o Repositório e Instalar Dependências
Clona ou atualiza (`git pull`) o repositório **JM3L0/flower_puro** e instala as dependências oficiais do Flower e PyTorch.

In [ ]:
import os

REPO_URL = "https://github.com/JM3L0/flower_puro.git"
REPO_DIR = "/content/flower_puro"

if not os.path.exists(REPO_DIR):
    print("Clonando repositório do GitHub...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Atualizando repositório via git pull...")
    %cd {REPO_DIR}
    !git reset --hard
    !git pull origin main

%cd {REPO_DIR}

print("\nInstalando dependências do Flower e PyTorch...")
!pip install -q datasets "flwr-datasets[vision]>=0.5.0" torch torchvision matplotlib numpy "pillow>=10.3.0,<12.0.0"
!pip install -q -e .
print("\n✔ AMBIENTE PRONTO PARA FLOWER PURO!")

## 3. Painel de Controle Interativo (Configuração Sem Código)
Ajuste os parâmetros da federação diretamente pelo formulário abaixo e execute a célula.

In [ ]:
#@title ⚙️ Painel de Controle da Simulação Federada { display-mode: "form" }
#@markdown ### 1. Estratégia de Agregação do Flower
estrategia_agregacao = "FedAvg" #@param ["FedAvg", "FedMedian", "FedProx"]

#@markdown ---
#@markdown ### 2. Hiperparâmetros do Treinamento Federado
num_rodadas = 10 #@param {type:"integer"}
num_clientes = 10 #@param {type:"integer"}
epocas_locais = 2 #@param {type:"integer"}
learning_rate = 0.01 #@param {type:"number"}
batch_size = 64 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}

#@markdown ---
#@markdown ### 3. Heterogeneidade dos Dados entre os Clientes (Dirichlet)
distribuicao_dados = "IID (Homogêneo - Alpha=100.0)" #@param ["IID (Homogêneo - Alpha=100.0)", "Não-IID Suave (Alpha=1.0)", "Não-IID Moderado (Alpha=0.5)", "Não-IID Severo (Alpha=0.1)"]

mapa_alpha = {
    "IID (Homogêneo - Alpha=100.0)": 100.0,
    "Não-IID Suave (Alpha=1.0)": 1.0,
    "Não-IID Moderado (Alpha=0.5)": 0.5,
    "Não-IID Severo (Alpha=0.1)": 0.1,
}
alpha_escolhido = mapa_alpha[distribuicao_dados]

import os

# Garante que o repositório esteja clonado caso a célula 2 tenha sido pulada
if not os.path.exists("/content/flower_puro"):
    print("⏳ Repositório não encontrado. Clonando do GitHub...")
    !git clone https://github.com/JM3L0/flower_puro.git /content/flower_puro
    %cd /content/flower_puro
    !pip install -q datasets "flwr-datasets[vision]>=0.5.0" torch torchvision matplotlib numpy "pillow>=10.3.0,<12.0.0"
    !pip install -q -e .
else:
    os.chdir("/content/flower_puro")

# Executa a simulação via subprocesso limpo do Python (isolamento total de memória)
cmd = (
    f"python simulacao_direta.py "
    f"--strategy {estrategia_agregacao} "
    f"--rounds {num_rodadas} "
    f"--clients {num_clientes} "
    f"--epochs {epocas_locais} "
    f"--lr {learning_rate} "
    f"--batch_size {batch_size} "
    f"--alpha {alpha_escolhido} "
    f"--seed {seed} "
    f"--output_dir resultados"
)
!{cmd}


## 4. Visualização dos Resultados e Métricas
Exibe as curvas de convergência de Acurácia Global, evolução da Perda (Loss) e tabelas consolidadas.

In [ ]:
import glob
import os
from IPython.display import display, Markdown, Image

resultados_dir = "/content/flower_puro/resultados"
tabela_md = os.path.join(resultados_dir, "TABELA_RESULTADOS.md")
grafico_png = os.path.join(resultados_dir, "curvas_convergencia.png")

# 1. Exibe a Tabela de Resultados do Aprendizado Federado
if os.path.exists(tabela_md):
    print("\n📋 TABELA DE RESULTADOS DO APRENDIZADO FEDERADO:\n")
    with open(tabela_md, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))

# 2. Exibe as Curvas de Convergência
if os.path.exists(grafico_png):
    print("\n📊 FIGURA DE CONVERGÊNCIA (ACURÁCIA E LOSS):\n")
    display(Image(filename=grafico_png, width=850))


## 5. Download dos Resultados (Sem Dependência de Google Drive)
Baixa um arquivo compactado `.zip` contendo os gráficos, arquivos de métricas em JSON e checkpoints dos modelos treinados diretamente pelo navegador.

In [ ]:
import os
import shutil
import datetime
from google.colab import files

resultados_dir = "/content/flower_puro/resultados"
timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_nome = f"/content/resultados_flower_puro_{timestamp}"

shutil.make_archive(zip_nome, "zip", resultados_dir)
print(f"[+] Resultados compactados: {zip_nome}.zip")
print("[*] Baixando arquivo para seu computador...")
files.download(f"{zip_nome}.zip")
